In [ ]:
import os
import sys
import json
import glob
import shutil
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = r"d:\New folder\Non-Invasive\rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.EfficientPhys import EfficientPhys

In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data", "Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data", "Headmotion", "groupB")

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- EfficientPhys preprocessing params -----
CHUNK_LENGTH = 180     # frames per clip
IMG_H, IMG_W = 72, 72
LABEL_TYPE   = "DiffNormalized"  # needs cumsum in post-processing
FRAME_DEPTH  = 10      # TSM temporal window size

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)

In [ ]:
# Read video frames

def read_video_frames(video_path):
    """Read all frames from an MP4 file.

    Returns:
        frames (np.ndarray): shape (T, H, W, 3), dtype uint8, RGB order.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)

In [ ]:
def read_ppg_synced(session_path, num_frames):
    """
    Reads the PPG signal from 'ppg.csv' and resamples it to match the exact 
    timestamps of the video frames from 'frame_timestamps.csv'.
    """
    import pandas as pd
    import numpy as np
    import os
    
    # 1. Read the video frame timestamps
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    # Validation
    if len(frame_df) != num_frames:
        print(f"Warning: Video has {num_frames} frames, but frame_timestamps.csv has {len(frame_df)} rows. Using min count.")
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    # 2. Read the raw PPG data
    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    # Clip frame times to valid ppg range to avoid extrapolation
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    
    # 3. Resample (Interpolate)
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)

In [ ]:
# Normalization functions

def standardized_data(data):
    """Standardized: global z-score over all pixels and frames."""
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data


def diff_normalize_label(label):
    """DiffNormalized label: finite difference normalised by std, zero-padded."""
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)

In [ ]:
# Face crop + resize

def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    """Detect face on frame 0, expand bbox by coef, resize all frames."""
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

In [ ]:
# Discover subjects and read ground truth heart rate

all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
print(f"Found {len(all_dirs)} subject folders\n")

subjects = []

for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")

    session_path = subj_dir

    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    
    if not video_files:
        print(f"No video found for {subj_id}, skipping.")
        continue
    video_path = video_files[0]

    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_path,
        "session_path": session_path,
    })
    print(f"  {subj_id}  video={os.path.basename(video_path)}")

print(f"\nTotal subjects: {len(subjects)}")

In [ ]:
# Data preprocessing

# Clear any previous preprocessed data
if os.path.exists(PREPROCESSED_PATH):
    shutil.rmtree(PREPROCESSED_PATH)
os.makedirs(PREPROCESSED_PATH)
print(f"Cleared and recreated: {PREPROCESSED_PATH}\n")

all_input_files = []

for subj in subjects:
    subj_key     = subj["subj_key"]
    video_path   = subj["video_path"]
    session_path = subj["session_path"]

    print(f"=== Processing {subj_key} ===")

    # Read video
    frames = read_video_frames(video_path)
    T = frames.shape[0]
    print(f"  Video: {T} frames @ {VIDEO_FPS} fps")

    # Read and sync PPG
    ppg_signal = read_ppg_synced(session_path, T)
    print(f"  PPG green: min={ppg_signal.min():.0f}, max={ppg_signal.max():.0f}")

    # Crop face and resize to 72x72
    frames_cropped = crop_face_resize(frames, IMG_H, IMG_W)

    # Standardized data (3 channels) -- input for EfficientPhys
    std_data = standardized_data(frames_cropped)  # (T, 72, 72, 3)

    # DiffNormalized label
    label = diff_normalize_label(ppg_signal)  # (T,)

    # Chunk into clips of CHUNK_LENGTH
    clip_num = T // CHUNK_LENGTH
    data_clips  = np.array([std_data[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH] for i in range(clip_num)])
    label_clips = np.array([label[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH]    for i in range(clip_num)])

    # Save per-subject subfolder
    subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
    os.makedirs(subj_dir)

    subj_files = []
    for chunk_idx in range(clip_num):
        input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.npy")
        label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")

        np.save(input_path, data_clips[chunk_idx])   # (CHUNK_LENGTH, 72, 72, 3)
        np.save(label_path, label_clips[chunk_idx])  # (CHUNK_LENGTH,)
        subj_files.append(input_path)

    all_input_files.extend(subj_files)
    print(f"  {clip_num} clips -> {subj_dir}\n")

print(f"Total clips saved: {len(all_input_files)}")
print("\nFolder structure:")
for subj in subjects:
    d = os.path.join(PREPROCESSED_PATH, subj["subj_key"])
    n = len(glob.glob(os.path.join(d, "*_input*.npy")))
    print(f"  {subj['subj_key']}/  ({n} clips)")

In [ ]:
# PyTorch Dataset + DataLoader

class EfficientPhysDataset(Dataset):

    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (D, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (D,)

        # NDHWC -> NDCHW
        data = np.transpose(data, (0, 3, 1, 2))  # (D, 3, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                      # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]   # +6 skips "_input"

        return data, label, subject_id, chunk_id


dataset = EfficientPhysDataset(all_input_files)
print(f"Dataset: {len(dataset)} clips")

# For training we should shuffle
loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)
print(f"DataLoader ready: {len(loader)} batches")

In [ ]:
# Training Initialization

EPOCHS = 10
LR = 1e-4

model = EfficientPhys(frame_depth=FRAME_DEPTH, img_size=IMG_H).to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0)

num_train_batches = len(loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=num_train_batches
)

print(f"Model total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training Loop

for epoch in range(EPOCHS):
    print(f"\n====Training Epoch: {epoch+1}/{EPOCHS}====")
    running_loss = 0.0
    model.train()
    
    tbar = tqdm(loader, ncols=80)
    for idx, batch in enumerate(tbar):
        data, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)
        
        N, D, C, H, W = data.shape
        data = data.view(N * D, C, H, W)
        labels = labels.view(-1, 1)
        
        # Trim to multiple of FRAME_DEPTH for TSM alignment
        trim_len = (N * D) // FRAME_DEPTH * FRAME_DEPTH
        data = data[:trim_len]
        labels = labels[:trim_len]
        
        # CRITICAL: Append one extra frame because EfficientPhys does
        # torch.diff internally, which reduces length by 1.
        # The trainer duplicates the last frame for padding.
        last_frame = data[-1:].clone()
        data = torch.cat([data, last_frame], dim=0)
        
        optimizer.zero_grad()
        pred_ppg = model(data)
        loss = criterion(pred_ppg, labels)
        loss.backward()
        
        optimizer.step()
        scheduler.step()
        
        running_loss += loss.item()
        tbar.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(loader)
    print(f"Epoch {epoch+1} Average Loss: {epoch_loss:.4f}")

In [ ]:
# Save the trained model weights

model_save_path = os.path.join(REPO_ROOT, "final_model_release", "GroupB_EfficientPhys.pth")
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

torch.save(model.state_dict(), model_save_path)
print(f"\nSaved final model weights to:\n{model_save_path}")